# Gu & Kurov (2020) Replication — Informational Role of Twitter Sentiment
**Ashley Thompson — Capstone**

Replication of: Gu & Kurov (2020), *Informational role of social media: Evidence from Twitter sentiment*, **Journal of Banking and Finance** 121, 105969.

### What this notebook does

| Paper Table | This Notebook |
|---|---|
| Table 2: Return predictability | Fama-MacBeth: `return_oo ~ twitter_sent_lag1 + controls` |
| Table 3: No-reversal test | FMB with 5 lags of Twitter sentiment |
| Table 4: Analyst coverage heterogeneity | **Omitted** — Bloomberg no longer provides analyst coverage counts |
| Table 5: Twitter vs. news sentiment | FMB with both sentiment types |
| Table 6: L-S trading strategy | Daily long top-decile / short bottom-decile Twitter sentiment |

**Key methodological difference from our existing FE analysis:** Gu & Kurov use Fama-MacBeth (1973) cross-sectional regressions with Newey-West SEs — **not panel FE**. The return measure is open-to-open (not open-to-close), because Bloomberg releases Twitter sentiment at 9:20am, allowing you to trade at the 9:30am open.

> **Before running:** Runtime → Change runtime type → T4 GPU (optional)

## 1. Install dependencies

In [ ]:
!pip install -q linearmodels pandas_datareader statsmodels

## 2. Load data

In [ ]:
# Option A — Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Capstone/'
DATA_PATH = DRIVE_DIR + ('panel_long_extended.csv' if os.path.exists(DRIVE_DIR + 'panel_long_extended.csv') else 'panel_long.csv')
print(f'Using: {DATA_PATH}')

In [ ]:
# GitHub output setup
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone', f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git', REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Imports and variable construction

In [ ]:
import warnings
import numpy as np
import pandas as pd
from linearmodels.panel import FamaMacBeth

warnings.filterwarnings('ignore')
np.random.seed(42)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')

HAS_SPREAD = 'bid_ask_spread' in long.columns

print(f'Bid-ask spread:   {"AVAILABLE" if HAS_SPREAD else "MISSING — omitted from controls"}')
print(f'Analyst coverage: unavailable from Bloomberg — Table 4 is omitted')

In [ ]:
# ── Open-to-open return (Gu & Kurov's dependent variable) ────────────────
# Return_i,t = (open_{t+1} - open_t) / open_t × 100
# Bloomberg releases Twitter sentiment at 9:20am → trade at 9:30am open
long['px_open_next'] = long.groupby('ticker')['px_open'].shift(-1)
long['return_oo']    = (long['px_open_next'] - long['px_open']) / long['px_open'] * 100

# ── Rogers-Satchell (1991) realized volatility ────────────────────────────
# Vol = (lnH-lnC)(lnH-lnO) + (lnL-lnC)(lnL-lnO), expressed in %
lH = np.log(long['px_high'].clip(lower=1e-8))
lL = np.log(long['px_low'].clip(lower=1e-8))
lO = np.log(long['px_open'].clip(lower=1e-8))
lC = np.log(long['px_close'].clip(lower=1e-8))
long['vol_rs'] = ((lH - lC)*(lH - lO) + (lL - lC)*(lL - lO)).clip(lower=0) * 100

# ── Abnormal trading volume ───────────────────────────────────────────────
# (volume_t - mean_volume_i) / mean_volume_i × 100
mv = long.groupby('ticker')['volume'].transform('mean')
long['abnorm_vol'] = ((long['volume'] - mv) / mv) * 100

# ── Firm size: log(market cap) ────────────────────────────────────────────
long['log_mkt_cap'] = np.log(long['mkt_cap'].clip(lower=1e-8))

# ── Lagged Twitter / news sentiment ──────────────────────────────────────
for k in range(1, 6):
    long[f'twitter_sent_lag{k}'] = long.groupby('ticker')['twitter_sent'].shift(k)
long['news_sent_lag1'] = long.groupby('ticker')['news_sent'].shift(1)

# ── 5-day rolling lags for all controls ──────────────────────────────────
ctrl_base = ['return_oo', 'abnorm_vol', 'vol_rs', 'log_mkt_cap']
if HAS_SPREAD:
    ctrl_base.append('bid_ask_spread')

for var in ctrl_base:
    for k in range(1, 6):
        long[f'{var}_lag{k}'] = long.groupby('ticker')[var].shift(k)

ctrl_str = ' + '.join(f'{v}_lag{k}' for v in ctrl_base for k in range(1, 6))
print(f'Controls: {ctrl_str[:120]}…')
print(f'return_oo: {long["return_oo"].notna().sum():,} non-missing obs')
print(f'vol_rs:    {long["vol_rs"].notna().sum():,} non-missing obs')

## 4. Fama-French Risk Adjustment (optional)
Downloads daily 4-factor Fama-French-Carhart loadings from Ken French's data library. Risk-adjusted returns = residuals from firm-level OLS on the 4 factors. If download fails, raw returns are used throughout.

In [ ]:
import statsmodels.api as sm
import pandas_datareader.data as web

ff_factors = None
HAS_ADJ_RETURN = False

try:
    start, end = long['date'].min(), long['date'].max()
    ff3   = web.DataReader('F-F_Research_Data_Factors_daily', 'famafrench', start=start, end=end)[0]
    mom   = web.DataReader('F-F_Momentum_Factor_daily',       'famafrench', start=start, end=end)[0]
    ff3.index = pd.to_datetime(ff3.index, format='%Y%m%d')
    mom.index = pd.to_datetime(mom.index, format='%Y%m%d')
    ff_factors = ff3.join(mom, how='inner') / 100
    ff_factors.columns = [c.strip() for c in ff_factors.columns]
    print(f'FF factors downloaded: {ff_factors.shape[0]} days, cols: {list(ff_factors.columns)}')

    # Risk-adjust each ticker
    merged = long.merge(ff_factors.rename(columns={'Mkt-RF':'mkt_rf','SMB':'smb','HML':'hml','Mom   ':'mom'}),
                        left_on='date', right_index=True, how='left')
    residuals = np.full(len(merged), np.nan)
    for ticker, grp in merged.groupby('ticker'):
        sub = grp[['return_oo','mkt_rf','smb','hml','mom']].dropna()
        if len(sub) < 30: continue
        X = sm.add_constant(sub[['mkt_rf','smb','hml','mom']])
        ols_res = sm.OLS(sub['return_oo'], X).fit()
        residuals[sub.index] = ols_res.resid.values
    long['return_oo_adj'] = residuals
    HAS_ADJ_RETURN = long['return_oo_adj'].notna().sum() > 1000
    print(f'Risk-adjusted returns: {long["return_oo_adj"].notna().sum():,} obs')

except Exception as e:
    print(f'FF download/computation failed: {e}')
    print('Using raw returns only.')

## 5. Table 2 — Return Predictability with Twitter Sentiment
Fama-MacBeth regression (Eq. 3 in Gu & Kurov):

$$\text{Return}_{i,t} = a + b \cdot \text{Twitter}_{i,t-1} + \sum_{k=1}^{5} c_k \text{Return}_{i,t-k} + \sum_{k=1}^{5} d_k \text{Volume}_{i,t-k} + \sum_{k=1}^{5} f_k \text{Volatility}_{i,t-k} + \sum_{k=1}^{5} g_k \text{Size}_{i,t-k} + \sum_{k=1}^{5} h_k \text{Spread}_{i,t-k} + \varepsilon_{i,t}$$

**Key finding in original:** b ≈ 0.136*** (raw), 0.143*** (risk-adj). Mean spread in returns between top and bottom decile ≈ 0.272% per day.

In [ ]:
def stars(p):
    if p < 0.01: return '***'
    if p < 0.05: return '**'
    if p < 0.1:  return '*'
    return ''

def fit_fmb(formula, data, label, bandwidth=5):
    """Fit Fama-MacBeth regression and print Twitter/news coefficients."""
    print(f'\n{"="*65}')
    print(f'  {label}')
    print('='*65)
    try:
        panel = data.set_index(['ticker', 'date']) if 'ticker' in data.columns else data
        mod = FamaMacBeth.from_formula(formula, data=panel)
        res = mod.fit(cov_type='kernel', bandwidth=bandwidth)
        for p in res.params.index:
            if any(k in p.lower() for k in ['twitter', 'news', 'hac']):
                c, s, pv = float(res.params[p]), float(res.std_errors[p]), float(res.pvalues[p])
                print(f'  {p:<45s}  {c:>10.6f}  ({s:.6f})  {stars(pv)}')
        print(f'  N = {res.nobs}')
        return res
    except Exception as e:
        print(f'  [ERROR] {e}')
        return None

panel_df = long.copy()

# Raw return, equal-weighted
t2_raw = fit_fmb(
    f'return_oo ~ twitter_sent_lag1 + {ctrl_str}',
    panel_df,
    'TABLE 2 (1): Raw return_oo, equal-weighted'
)

# Risk-adjusted return (if available)
t2_adj = None
if HAS_ADJ_RETURN:
    adj_ctrl = ctrl_str.replace('return_oo_', 'return_oo_adj_')
    for k in range(1, 6):
        panel_df[f'return_oo_adj_lag{k}'] = panel_df.groupby('ticker')['return_oo_adj'].shift(k)
    t2_adj = fit_fmb(
        f'return_oo_adj ~ twitter_sent_lag1 + {adj_ctrl}',
        panel_df,
        'TABLE 2 (2): Risk-adjusted return, equal-weighted'
    )

## 6. Table 3 — No-Reversal Test
Includes 5 lags of Twitter sentiment simultaneously. Under the information hypothesis (Twitter reveals fundamentals), **only lag 1 should be significant**; lags 2–5 near zero indicate no price reversal. Significant lags 2–5 would suggest Twitter simply shifts noise-trader demand (temporary price pressure) rather than conveying value-relevant information.

In [ ]:
multi_lag_sent = ' + '.join(f'twitter_sent_lag{k}' for k in range(1, 6))

t3_raw = fit_fmb(
    f'return_oo ~ {multi_lag_sent} + {ctrl_str}',
    panel_df,
    'TABLE 3 (1): No-reversal test — 5 Twitter lags, raw return'
)

t3_adj = None
if HAS_ADJ_RETURN:
    t3_adj = fit_fmb(
        f'return_oo_adj ~ {multi_lag_sent} + {adj_ctrl}',
        panel_df,
        'TABLE 3 (2): No-reversal test — 5 Twitter lags, risk-adjusted'
    )

print('\n  Interpretation:')
print('  Only lag 1 significant → permanent information effect (Gu & Kurov finding)')
print('  All lags significant / lags 2-5 significant → reverse causality or momentum')

## 7. Table 4 — Analyst Coverage Heterogeneity *(Omitted)*

Bloomberg no longer provides analyst coverage counts (`NUM_ANALYST`). This table is omitted from the replication. The remaining tables (5 and 6) are unaffected.

## 8. Table 5 — Twitter vs. News Sentiment
Tests whether Twitter provides incremental predictive power beyond news. Gu & Kurov find both remain significant (corr ≈ 0.20), suggesting Twitter conveys orthogonal information to traditional media.

In [ ]:
# News sentiment only
t5_news = fit_fmb(
    f'return_oo ~ news_sent_lag1 + {ctrl_str}',
    panel_df,
    'TABLE 5 (1): News sentiment only'
)

# Both Twitter and news
t5_both = fit_fmb(
    f'return_oo ~ twitter_sent_lag1 + news_sent_lag1 + {ctrl_str}',
    panel_df,
    'TABLE 5 (2): Twitter + news sentiment'
)

corr = long[['twitter_sent','news_sent']].corr().iloc[0,1]
print(f'\n  Contemporaneous corr(twitter_sent, news_sent) = {corr:.3f}')
print(f'  Gu & Kurov report ≈ 0.20')

## 9. Table 6 — Long-Short Trading Strategy
Daily portfolio: long top-decile Twitter sentiment firms, short bottom-decile. Hold for 24h. Gu & Kurov report annualized Sharpe ratio of 3.17 and ~24% excess return (before transaction costs) over Jan 2015–Feb 2017.

In [ ]:
# Assign decile ranks cross-sectionally each day
daily_decile = long.copy()
daily_decile['decile'] = daily_decile.groupby('date')['twitter_sent'].transform(
    lambda x: pd.qcut(x, q=10, labels=False, duplicates='drop')
)

# Daily L-S return
daily_ls = (
    daily_decile[daily_decile['decile'].isin([0, 9])]
    .groupby(['date', 'decile'])['return_oo']
    .mean()
    .unstack('decile')
    .rename(columns={0: 'short', 9: 'long'})
    .dropna()
)
daily_ls['ls_return'] = daily_ls['long'] - daily_ls['short']

n_days     = len(daily_ls)
mean_daily = daily_ls['ls_return'].mean()
std_daily  = daily_ls['ls_return'].std()
ann_return = mean_daily * 252
sharpe     = (mean_daily / std_daily) * np.sqrt(252) if std_daily > 0 else np.nan
win_rate   = (daily_ls['ls_return'] > 0).mean()

print(f'Trading strategy results:')
print(f'  N trading days:        {n_days}')
print(f'  Mean daily L-S return: {mean_daily:.4f}%')
print(f'  Annualized return:     {ann_return:.2f}%')
print(f'  Annualized Sharpe:     {sharpe:.2f}  (Gu & Kurov: 3.17)')
print(f'  Win rate:              {win_rate:.1%}')
print()
print('  Note: these figures are before transaction costs.')
print('  Gu & Kurov sample: Jan 2015–Feb 2017 (537 trading days).')

## 10. Export — Publication-ready LaTeX tables

In [ ]:
def fmb_coef(res, param):
    try:
        return float(res.params[param]), float(res.std_errors[param]), float(res.pvalues[param])
    except Exception:
        return float('nan'), float('nan'), 1.0

def multi_col_latex(col_models, col_labels, row_params, caption, label_str, notes=''):
    ncols = len(col_models)
    def fmt(res, p, se=False):
        if res is None or p not in res.params.index: return '---'
        c, s, pv = fmb_coef(res, p)
        return f'({s:.6f})' if se else f'{c:.6f}{stars(pv)}'
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        f'\\caption{{{caption}}}', f'\\label{{{label_str}}}',
        f'\\begin{{tabular}}{{l{"c"*ncols}}}', r'\hline\hline',
        ' & ' + ' & '.join(f'({i+1}) {col_labels[i]}' for i in range(ncols)) + r' \\',
        r'\hline',
    ]
    for p in row_params:
        pl = p.replace('_', r'\_')
        lines.append(pl + ' & ' + ' & '.join(fmt(m, p) for m in col_models) + r' \\')
        lines.append('& ' + ' & '.join(fmt(m, p, se=True) for m in col_models) + r' \\')
    lines += [
        r'\hline',
        'Estimator & ' + ' & '.join(['Fama-MacBeth']*ncols) + r' \\',
        'SE & ' + ' & '.join(['Newey-West (5)']*ncols) + r' \\',
        r'\hline\hline',
    ]
    if notes:
        lines.append(f'\\multicolumn{{{ncols+1}}}{{l}}{{\\footnotesize \\textit{{Notes:}} {notes}}} \\\\')
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

# Table 2
models_t2  = [t2_raw, t2_adj] if t2_adj else [t2_raw]
labels_t2  = ['Raw Return', 'Risk-Adj'] if t2_adj else ['Raw Return']
spread_note = 'bid-ask spread included.' if HAS_SPREAD else 'bid-ask spread omitted (pull from Bloomberg).'

tex_t2 = multi_col_latex(
    models_t2, labels_t2, ['twitter_sent_lag1'],
    caption='Gu \\& Kurov (2020) Replication --- Table 2: Return Predictability',
    label_str='tab:gk_t2',
    notes=f'Fama-MacBeth. Newey-West SEs (5 lags). Open-to-open return. Controls: 5 lags of return, abnorm\\_vol, vol\\_rs, log\\_mkt\\_cap, {spread_note} *** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
)

# Table 3
models_t3 = [t3_raw, t3_adj] if t3_adj else [t3_raw]
labels_t3 = ['Raw Return', 'Risk-Adj'] if t3_adj else ['Raw Return']
tex_t3 = multi_col_latex(
    models_t3, labels_t3, [f'twitter_sent_lag{k}' for k in range(1,6)],
    caption='Gu \\& Kurov (2020) Replication --- Table 3: No-Reversal Test',
    label_str='tab:gk_t3',
    notes='Only lag 1 significant ⟹ permanent information effect. *** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
)

# Table 5
tex_t5 = multi_col_latex(
    [t5_news, t5_both], ['News Only', 'Twitter + News'],
    ['twitter_sent_lag1', 'news_sent_lag1'],
    caption='Gu \\& Kurov (2020) Replication --- Table 5: Twitter vs. News Sentiment',
    label_str='tab:gk_t5',
    notes='Both sentiments use 1-day lag. Low correlation ⟹ orthogonal information. *** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
)

# Table 6
ls_tex_rows = [
    r'Mean daily L-S return (\%) & ' + f'{mean_daily:.4f}' + r' \\',
    r'Annualized return (\%) & '     + f'{ann_return:.2f}' + r' \\',
    r'Annualized Sharpe & '          + f'{sharpe:.2f}'     + r' \\',
    r'Win rate & '                   + f'{win_rate:.1%}'   + r' \\',
    r'Trading days & '               + f'{n_days}'         + r' \\',
]
tex_t6 = (
    r'\begin{table}[htbp]' + '\n' + r'\centering' + '\n'
    r'\caption{Gu \& Kurov (2020) Replication --- Table 6: Long-Short Strategy}' + '\n'
    r'\label{tab:gk_t6}' + '\n'
    r'\begin{tabular}{lc}' + '\n' + r'\hline\hline' + '\n'
    r'Statistic & Value \\' + '\n' + r'\hline' + '\n'
    + '\n'.join(ls_tex_rows) + '\n'
    r'\hline\hline' + '\n'
    r'\multicolumn{2}{l}{\footnotesize \textit{Notes:} Long (short) = top (bottom) decile Twitter sentiment. 24h hold.} \\' + '\n'
    r'\end{tabular}' + '\n' + r'\end{table}'
)

for fname, content in [('gk_t2.tex', tex_t2), ('gk_t3.tex', tex_t3),
                       ('gk_t5.tex', tex_t5), ('gk_t6.tex', tex_t6)]:
    with open(OUTPUT_PATH + fname, 'w') as f:
        f.write(content)
    print(f'Saved {OUTPUT_PATH + fname}')

print(tex_t2)

In [ ]:
# Push outputs to GitHub
import os, subprocess
os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])
staged = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(['git', 'log', 'origin/main..HEAD', '--oneline'], capture_output=True, text=True)
if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add Gu & Kurov replication results'], check=True)
if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit.')